# Intel Image Classification (4 Classes)
**Author:** Muhammad Akbar Pradana  
**Email:** akbarprdn2512@gmail.com  
**Dicoding ID:** akbarprdna  

Transfer learning with **EfficientNetB0** to classify natural scenes into 4 classes: `buildings`, `forest`, `mountain`, `sea`.

## 1. Import Libraries

In [ ]:
# Standard & third-party imports
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU Available      : {tf.config.list_physical_devices("GPU")}')

## 2. Configuration & Paths

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR        = r'c:/Users/akbar/VSCode Project/intel-image-4class'
DATA_DIR        = os.path.join(BASE_DIR, 'data', 'intel-image-classification')
FIGURES_DIR     = os.path.join(BASE_DIR, 'figures')
MODELS_DIR      = os.path.join(BASE_DIR, 'models')
SAVED_MODEL_DIR = os.path.join(MODELS_DIR, 'saved_model')
TFLITE_DIR      = os.path.join(MODELS_DIR, 'tflite')
TFJS_DIR        = os.path.join(MODELS_DIR, 'tfjs_model')
CHECKPOINT_PATH = os.path.join(MODELS_DIR, 'best_model.keras')

# ── Hyper-parameters ────────────────────────────────────────────────────────
SELECTED_CLASSES = ['forest', 'sea', 'mountain', 'buildings']
IMG_SIZE         = (224, 224)
BATCH_SIZE       = 32
EPOCHS_FROZEN    = 10   # train only classifier head
EPOCHS_FINETUNE  = 15   # unfreeze top layers
SEED             = 42

# Create output directories
for d in [FIGURES_DIR, MODELS_DIR, SAVED_MODEL_DIR, TFLITE_DIR, TFJS_DIR]:
    os.makedirs(d, exist_ok=True)

print('All directories ready.')

## 3. Data Preparation

### 3.1 Download Dataset from Kaggle

In [ ]:
# Load Kaggle credentials from kaggle.json (never hard-code API keys in notebooks!)
kaggle_json_path = os.path.join(BASE_DIR, 'kaggle.json')
kaggle_cfg_dir   = os.path.expanduser('~/.kaggle')

os.makedirs(kaggle_cfg_dir, exist_ok=True)

# Copy credentials only if the file exists
if os.path.exists(kaggle_json_path):
    import shutil
    dest = os.path.join(kaggle_cfg_dir, 'kaggle.json')
    shutil.copy2(kaggle_json_path, dest)
    os.chmod(dest, 0o600)
    print('Kaggle credentials configured.')
else:
    print('WARNING: kaggle.json not found — skipping credential setup.')

# Download & unzip (skip if already present)
zip_flag = os.path.join(DATA_DIR, '..', 'intel-image-classification.zip')
if not os.path.isdir(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    !kaggle datasets download -d puneet6060/intel-image-classification \\
        --unzip -p "{os.path.dirname(DATA_DIR)}"
    print('Dataset downloaded.')
else:
    print('Dataset already present — skipping download.')

### 3.2 Build DataFrame

In [ ]:
def build_dataframe(split_name: str, classes: list) -> pd.DataFrame:
    """Walk <split>/<split>/<class>/ and return a tidy DataFrame."""
    split_dir = os.path.join(DATA_DIR, split_name, split_name)
    rows = []
    for cls in os.listdir(split_dir):
        if cls not in classes:
            continue
        cls_dir = os.path.join(split_dir, cls)
        for img_name in os.listdir(cls_dir):
            rows.append({'path': os.path.join(cls_dir, img_name), 'label': cls})
    return pd.DataFrame(rows)

# Use only seg_train; split manually for train / val / test
all_df = build_dataframe('seg_train', SELECTED_CLASSES)
print(f'Total images : {len(all_df)}')
print(f'Classes      : {sorted(all_df["label"].unique())}')
print(f'\nClass distribution:\n{all_df["label"].value_counts()}')

### 3.3 Visualize Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=all_df, x='label',
              order=all_df['label'].value_counts().index, ax=ax)
ax.set_title('Class Distribution — Intel Image Classification', fontsize=13)
ax.set_xlabel('Class'); ax.set_ylabel('Count')
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'class_distribution.png'), dpi=150)
plt.show()
print('Figure saved.')

### 3.4 Sample Images per Class

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, cls in enumerate(SELECTED_CLASSES):
    cls_paths = all_df[all_df['label'] == cls]['path'].sample(2, random_state=SEED).values
    for row, path in enumerate(cls_paths):
        ax = axes[row, col]
        ax.imshow(mpimg.imread(path))
        ax.set_title(cls if row == 0 else '', fontsize=11)
        ax.axis('off')
fig.suptitle('Sample Images per Class', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'sample_images.png'), dpi=150)
plt.show()

### 3.5 Train / Validation / Test Split

In [ ]:
# 70 % train | 15 % val | 15 % test  (stratified)
train_df, temp_df = train_test_split(
    all_df, test_size=0.30, random_state=SEED, stratify=all_df['label'])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label'])

print(f'Train  : {len(train_df):>5} images')
print(f'Val    : {len(val_df):>5} images')
print(f'Test   : {len(test_df):>5} images')

### 3.6 Data Generators

In [ ]:
# Training generator with augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
# Validation / test generators — preprocessing only
eval_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

def make_gen(datagen, df, shuffle=True):
    return datagen.flow_from_dataframe(
        df, x_col='path', y_col='label',
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=shuffle, seed=SEED
    )

train_gen = make_gen(train_datagen, train_df, shuffle=True)
val_gen   = make_gen(eval_datagen,  val_df,   shuffle=False)
test_gen  = make_gen(eval_datagen,  test_df,  shuffle=False)

class_names = list(train_gen.class_indices.keys())
print('Class indices:', train_gen.class_indices)

## 4. Modelling

### 4.1 Build Model

In [ ]:
def build_model(num_classes: int = 4) -> tf.keras.Model:
    """EfficientNetB0 backbone with a custom classification head."""
    backbone = EfficientNetB0(
        include_top=False, weights='imagenet',
        input_shape=(*IMG_SIZE, 3)
    )
    backbone.trainable = False  # freeze backbone initially

    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='EfficientNetB0_4Class')

model = build_model(num_classes=len(class_names))
model.summary()

### 4.2 Callbacks

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(CHECKPOINT_PATH, monitor='val_accuracy',
                     save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                       patience=3, min_lr=1e-6, verbose=1)
]

### 4.3 Phase 1 — Train Classifier Head (backbone frozen)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_frozen = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FROZEN,
    callbacks=callbacks,
    verbose=1
)

### 4.4 Phase 2 — Fine-tune Top Layers

In [ ]:
# Unfreeze the last 30 layers of the backbone
model.layers[1].trainable = True  # backbone is layer index 1
for layer in model.layers[1].layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FINETUNE,
    callbacks=callbacks,
    verbose=1
)

## 5. Evaluation & Visualisation

### 5.1 Training Curves

In [ ]:
def plot_history(h_frozen, h_ft):
    """Combine frozen + fine-tune histories and plot accuracy/loss curves."""
    acc  = h_frozen.history['accuracy']      + h_ft.history['accuracy']
    val_acc = h_frozen.history['val_accuracy'] + h_ft.history['val_accuracy']
    loss = h_frozen.history['loss']           + h_ft.history['loss']
    val_loss = h_frozen.history['val_loss']   + h_ft.history['val_loss']
    epochs = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, acc,     label='Train Acc')
    ax1.plot(epochs, val_acc, label='Val Acc')
    ax1.set_title('Accuracy'); ax1.legend(); ax1.set_xlabel('Epoch')

    ax2.plot(epochs, loss,     label='Train Loss')
    ax2.plot(epochs, val_loss, label='Val Loss')
    ax2.set_title('Loss'); ax2.legend(); ax2.set_xlabel('Epoch')

    plt.suptitle('Training Curves', fontsize=13)
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, 'training_curves.png'), dpi=150)
    plt.show()

plot_history(history_frozen, history_ft)

### 5.2 Classification Report & Confusion Matrix

In [ ]:
test_gen.reset()
y_pred_proba = model.predict(test_gen, verbose=1)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = test_gen.classes

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix — Test Set')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 6. Model Conversion & Export

### 6.1 SavedModel

In [ ]:
model.export(SAVED_MODEL_DIR)
print(f'SavedModel saved to: {SAVED_MODEL_DIR}')

### 6.2 TF-Lite (dynamic-range quantisation)

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(TFLITE_DIR, 'model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

# Save class labels alongside the TFLite model
with open(os.path.join(TFLITE_DIR, 'label.txt'), 'w') as f:
    f.write('\n'.join(class_names))

print(f'TFLite model saved ({len(tflite_model)/1024/1024:.2f} MB) -> {tflite_path}')
print(f'Labels: {class_names}')

### 6.3 TensorFlow.js

In [ ]:
sys.argv = [
    'tensorflowjs_converter',
    '--input_format=tf_saved_model',
    '--output_format=tfjs_graph_model',
    SAVED_MODEL_DIR,
    TFJS_DIR
]
from tensorflowjs.converters.converter import convert
convert(sys.argv[1:])
print(f'TFJS model saved to: {TFJS_DIR}')

## 7. Reload Model (Kernel-Restart Safe)

In [ ]:
# Run this cell to restore model & class names after a kernel restart
# without re-training from scratch.

if os.path.exists(CHECKPOINT_PATH):
    model = keras.models.load_model(CHECKPOINT_PATH)
    print(f'Model loaded from checkpoint: {CHECKPOINT_PATH}')
elif os.path.isdir(SAVED_MODEL_DIR):
    model = keras.models.load_model(SAVED_MODEL_DIR)
    print(f'Model loaded from SavedModel: {SAVED_MODEL_DIR}')
else:
    print('No saved model found — please run training cells first.')

# Restore class names from label.txt
label_path = os.path.join(TFLITE_DIR, 'label.txt')
if os.path.exists(label_path):
    with open(label_path) as f:
        class_names = f.read().splitlines()
    print(f'Class names restored: {class_names}')

## 8. Inference Demo

In [ ]:
import random
from PIL import Image

def predict_single(img_path: str) -> dict:
    """Run inference on a single image and return class probabilities."""
    img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
    arr = np.expand_dims(np.array(img, dtype='float32'), 0)
    arr = preprocess_input(arr)
    probs = model.predict(arr, verbose=0)[0]
    return {cls: float(p) for cls, p in zip(class_names, probs)}

# Pick one random test image per class
fig, axes = plt.subplots(1, len(class_names), figsize=(14, 4))
for ax, cls in zip(axes, class_names):
    sample_path = test_df[test_df['label'] == cls]['path'].sample(1, random_state=SEED).values[0]
    preds = predict_single(sample_path)
    top_cls = max(preds, key=preds.get)
    color = 'green' if top_cls == cls else 'red'
    ax.imshow(mpimg.imread(sample_path))
    ax.set_title(f'True: {cls}\nPred: {top_cls} ({preds[top_cls]:.0%})',
                 color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('Inference Demo', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'inference_demo.png'), dpi=150)
plt.show()